# Week 2 Session 3: Evaluation Metrics for Risk Prediction

In Session 2, we built baseline models with our 6 engineered features and demonstrated that accuracy is misleading for imbalanced data. Now we'll implement proper evaluation metrics for clinical risk prediction.

---

## Learning Objectives

| Objective | Description |
|-----------|-------------|
| **ROC-AUC vs PR-AUC** | Understand why PR-AUC is more informative than ROC-AUC for imbalanced datasets |
| **Precision at Fixed Recall** | Calculate precision at specific recall thresholds for clinical threshold selection |
| **Business Impact Metrics** | Develop cost-benefit analysis and alerts per prevented event metrics |

---

## Why Standard Accuracy Fails in Clinical Risk Prediction

From Session 2, we learned:
- Our data has severe **class imbalance** (only ~3% positive cases)
- Majority class baseline achieves **~97% accuracy** but catches **0% of high-risk patients**
- We need metrics that focus on the minority class performance

**Clinical Context**: In healthcare, the cost of missing a high-risk patient (false negative) is typically much higher than the cost of reviewing a false alert (false positive). Standard accuracy treats both errors equally, making it unsuitable for clinical decision support.

In homework, you'll apply these evaluation techniques to all 22 features.

---

## Setup

We'll load the same libraries as Session 2, plus specialized metrics from scikit-learn.

**New imports:**
- `roc_curve`, `precision_recall_curve`: For plotting threshold-based performance curves
- `auc`, `average_precision_score`: For calculating area under curves
- Additional classification metrics for comprehensive evaluation

**Key design pattern**: We use `REPO_PATH` to ensure paths work regardless of where the notebook is executed.

## Setting Environment Up for Colab

Mount Google Drive and set the repository path so this notebook can access the EHR data and source code.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

### Project Configuration

This notebook loads paths from a YAML config file instead of hard-coded paths.

**Setup (one-time):**
1. In Colab's left sidebar, click the **Key** icon (Secrets)
2. Add a secret named `PROJECT_CONFIG_PATH`
3. Set the value to your config file path (e.g., `/content/drive/MyDrive/Project/config.yaml`)
4. Toggle "Notebook access" ON

In [ ]:
from google.colab import userdata
import yaml

try:
    config_path = userdata.get('PROJECT_CONFIG_PATH')
except:
    config_path = input("Enter path to your config.yaml: ")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REPO_PATH = config['repo_path']
print(f"Repository path: {REPO_PATH}")

### Install Source Code as Package

`pip install` installs the local source code as a Python package so you can import directly from it (e.g., `from week_2.helpers import get_col`). Re-run this cell after making changes to the source code.

In [ ]:
!pip install {REPO_PATH}/src_solutions/ -q

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os

print("Libraries loaded")

---

## Load Training Data

We load the `classifier_training_data.csv` that we generated in Session 1. This dataset contains:
- **6 engineered features** (temporal, clinical, demographics)
- **Daily instances** for each diabetic patient in the cohort
- **Binary label**: `will_have_high_risk_event_next_30d`

We'll also check the positive rate to understand the class imbalance we're dealing with.

**Data quality step**: We drop rows with NaN labels, as these represent instances where we cannot determine the outcome (e.g., patients with less than 30 days of follow-up).

In [ ]:
DATA_DIR = os.path.join(REPO_PATH, "data/week_2")
classifier_data = pd.read_csv(os.path.join(DATA_DIR, "classifier_training_data.csv"), low_memory=False)

label_col = 'will_have_high_risk_event_next_30d'
patient_col = 'patient_id'

valid_mask = classifier_data[label_col].notna()
classifier_data = classifier_data[valid_mask].copy()
classifier_data[label_col] = classifier_data[label_col].astype(int)

SESSION_1_FEATURES = [
    'days_since_last_hba1c',
    'current_hba1c_level',
    'encounters_last_90d',
    'age_at_date',
    'current_systolic_bp',
    'current_egfr'
]

feature_cols = [col for col in SESSION_1_FEATURES if col in classifier_data.columns]

print(f"Dataset loaded: {len(classifier_data):,} instances")
print(f"Features: {len(feature_cols)}")
print(f"Unique patients: {classifier_data[patient_col].nunique()}")
print(f"\nClass distribution:")
print(classifier_data[label_col].value_counts().to_string())
print(f"\nPositive rate: {classifier_data[label_col].mean():.2%}")

---

## Feature Overview

Let's examine the 6 features we'll use for prediction. These were engineered in Session 1 and represent different aspects of patient risk:

**Temporal features**: `days_since_last_hba1c` (monitoring adherence)

**Clinical features**: `current_hba1c_level`, `current_systolic_bp`, `current_egfr` (disease control)

**Utilization features**: `encounters_last_90d` (healthcare engagement)

**Demographics**: `age_at_date` (baseline risk)

**Note**: Coverage percentages indicate how often we have measurements for each feature. Missing values will be filled with 0 for logistic regression.

In [ ]:
print("Features available for prediction:")
print("=" * 60)
for i, col in enumerate(feature_cols, 1):
    non_null = classifier_data[col].notna().sum()
    coverage = 100 * non_null / len(classifier_data)
    print(f"{i:2}. {col:<35} ({coverage:.1f}% coverage)")
print(f"\nTotal features: {len(feature_cols)}")

---

## Patient-Level Train/Test Split

**Critical**: We must split by **patient**, not by row. This is essential for valid evaluation.

**Why patient-level splitting matters:**

| Issue | Consequence | Solution |
|-------|-------------|----------|
| **Temporal correlation** | A patient's day-100 features are highly correlated with day-101 | Keep all instances from one patient together |
| **Data leakage** | Model memorizes patient-specific patterns instead of generalizable risk factors | `GroupShuffleSplit` ensures all patient data stays in one split |
| **Real-world validity** | In production, new patients won't have historical data in training set | Simulates deployment on unseen patient population |

`GroupShuffleSplit` ensures all instances from one patient stay together in either train OR test, preventing information leakage.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

X = classifier_data[feature_cols].fillna(0)
y = classifier_data[label_col]
groups = classifier_data[patient_col]

for train_idx, test_idx in gss.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    test_data = classifier_data.iloc[test_idx].copy()

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Test positive rate: {y_test.mean():.2%}")

---

## Train a Model for Evaluation

We train a simple logistic regression model to demonstrate the evaluation metrics. The model itself isn't the focus - the **evaluation methodology** is.

**Key settings:**
- `StandardScaler`: Standardizes features to mean 0, std 1 (as introduced in Session 2)
- `class_weight='balanced'`: Automatically upweights the minority class to handle imbalance
- `fillna(0)`: Handle missing values (tree models handle this naturally, but linear models need clean data)
- `random_state=42`: Ensures reproducibility

We generate both **hard predictions** (`y_pred`) and **probability scores** (`y_prob`) because different metrics use different outputs:
- Hard predictions (0/1): Used for confusion matrix, precision, recall
- Probability scores (0-1): Used for ROC-AUC, PR-AUC, threshold selection

In [ ]:
# fillna(0) handles missing values for logistic regression (tree models handle NaN natively)
X_train_clean = X_train.fillna(0)
X_test_clean = X_test.fillna(0)

# Standardize features (as introduced in Session 2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred = lr_model.predict(X_test_scaled)
y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print(f"Model trained. Predictions generated for {len(y_test):,} test instances.")

---

# Classification Metrics for Imbalanced Data

## ROC-AUC (Receiver Operating Characteristic)

ROC-AUC measures the model's ability to **rank** positive instances higher than negative instances. It answers: "If I pick a random positive and random negative, what's the probability the model scores the positive higher?"

**Interpretation guidelines:**

| ROC-AUC | Interpretation |
|---------|----------------|
| 0.5 | Random guessing - no discriminative power |
| 0.7-0.8 | Acceptable performance |
| 0.8-0.9 | Good performance |
| 0.9+ | Excellent performance |

**Limitation for imbalanced data**: ROC-AUC can be overly optimistic because it uses False Positive Rate (FPR), which has a huge denominator (all true negatives). A model can have high ROC-AUC while generating many false alarms.

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"\nInterpretation: The model correctly ranks a random positive")
print(f"above a random negative {roc_auc:.1%} of the time.")

---

## Visualize ROC Curve

The ROC curve plots **True Positive Rate (TPR)** vs **False Positive Rate (FPR)** at every possible threshold.

**How to read it:**
- **Diagonal line (AUC=0.5)**: Random classifier - no discriminative power
- **Upper-left corner**: Perfect classifier - 100% TPR with 0% FPR
- **Area under curve (AUC)**: Summary statistic - higher is better

**Clinical interpretation**: Each point on the curve represents a different risk threshold. Moving up-right on the curve means catching more high-risk patients (higher TPR) but also generating more false alarms (higher FPR).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve', fontsize=14)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

## PR-AUC (Precision-Recall AUC)

**Note**: In scikit-learn, `average_precision_score` computes Average Precision (AP), which is equivalent to PR-AUC. We use the terms interchangeably throughout this notebook.

For imbalanced data, **PR-AUC is more informative than ROC-AUC**. It focuses on the positive class and shows the tradeoff between precision and recall.

**Key insight**: A random classifier has PR-AUC equal to the positive class rate (e.g., ~3% in our case), not 0.5 like ROC-AUC.

**Why PR-AUC is better for imbalanced data:**

| Metric | Focus | Baseline | Best for |
|--------|-------|----------|----------|
| ROC-AUC | Overall ranking ability | 0.5 | Balanced datasets |
| PR-AUC | Positive class performance | Positive rate | Imbalanced datasets |

**Clinical relevance**: PR-AUC directly measures how well the model identifies high-risk patients (recall) while minimizing false alarms (precision).

In [ ]:
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
baseline_pr = y_test.mean()

print(f"PR-AUC: {pr_auc:.4f}")
print(f"Random baseline PR-AUC: {baseline_pr:.4f}")
print(f"\nImprovement over random: {pr_auc / baseline_pr:.1f}x")

---

## Visualize Precision-Recall Curve

The PR curve plots **Precision** vs **Recall** at every threshold.

**Why PR > ROC for imbalanced data:**
- PR curve focuses entirely on the **positive class**
- Precision = TP / (TP + FP) - directly measures the fraction of alerts that are true positives
- The baseline is the positive class rate (not 0.5), so improvements are clearly visible

**How to read it:**
- **Upper-right corner**: Perfect classifier (100% precision and recall)
- **Flat line at class rate**: Random classifier baseline
- **Steep drop**: Model struggles to maintain precision as recall increases

**Clinical interpretation**: The curve shows the fundamental tradeoff - to catch more high-risk patients (higher recall), we must accept more false alarms (lower precision).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(recall_curve, precision_curve, 'b-', linewidth=2, 
        label=f'Logistic Regression (AP = {pr_auc:.4f})')
ax.axhline(y=baseline_pr, color='k', linestyle='--', linewidth=1,
           label=f'Random baseline (AP = {baseline_pr:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, max(precision_curve) * 1.1])

plt.tight_layout()
plt.show()

---

# Precision at Fixed Recall

In clinical settings, we often need to answer:
> "To catch X% of high-risk patients, how many false alarms will we have?"

This is **Precision at a fixed Recall level**.

**Clinical framing**: Healthcare organizations typically start with a minimum acceptable recall (sensitivity) - e.g., "We must catch at least 80% of high-risk patients" - and then ask what precision (positive predictive value) the model achieves at that operating point.

This metric directly answers stakeholder questions about alert workload.

In [ ]:
def precision_at_recall(y_true, y_prob, target_recall):
    """
    Find precision at a target recall level.
    
    Returns:
        precision: Precision at the target recall
        threshold: Probability threshold that achieves this recall
    """
    precision_vals, recall_vals, thresholds = precision_recall_curve(y_true, y_prob)
    # Note: thresholds has one fewer element than precision/recall arrays
    
    # Find threshold that gives at least target_recall
    # recall_vals is in decreasing order, so we find first value >= target
    for i in range(len(recall_vals) - 1, -1, -1):
        if recall_vals[i] >= target_recall:
            return precision_vals[i], thresholds[i] if i < len(thresholds) else 0.0
    
    return precision_vals[0], thresholds[0]

print("precision_at_recall function defined")

---

## Precision at Key Recall Levels

In clinical settings, stakeholders often ask: **"To catch 90% of high-risk patients, how many false alarms will we have?"**

This table answers that question:

| Column | Meaning |
|--------|----------|
| **Recall Target** | The sensitivity level we require (e.g., catch 90% of events) |
| **Precision** | What fraction of alerts are true positives at that recall |
| **Threshold** | The probability cutoff that achieves this recall |
| **Alerts/TP** | How many total alerts per true positive (1/Precision) |

This helps clinical teams understand the workload tradeoff: higher recall = more false alarms.

In [ ]:
recall_targets = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

print("Precision @ Fixed Recall:")
print("=" * 60)
print(f"{'Recall Target':<15} {'Precision':<12} {'Threshold':<12} {'Alerts/TP':<12}")
print("-" * 60)

for target in recall_targets:
    prec, thresh = precision_at_recall(y_test, y_prob, target)
    alerts_per_tp = 1 / prec if prec > 0 else float('inf')
    print(f"{target:<15.0%} {prec:<12.4f} {thresh:<12.4f} {alerts_per_tp:<12.1f}")

---

## Interpreting Precision at Recall Results

The table above shows the fundamental tradeoff in clinical decision support:

- To catch **90% of high-risk patients** (recall=0.90), we must accept many alerts per true positive
- This means clinical staff must review many patients to find each true high-risk case
- If that's too many false alarms, we can accept lower recall (miss some cases) to reduce alert fatigue

**Business decision**: What's the cost of missing a high-risk patient vs. the cost of reviewing false alerts?

**Clinical workflow consideration**: At 90% recall, precision drops to ~0.04, meaning clinical staff review ~22 alerts for every 1 true high-risk patient. This level of alert fatigue can lead to:
- Decreased trust in the system
- Incomplete follow-up on alerts
- Staff burnout

Therefore, threshold selection must balance clinical risk tolerance with operational feasibility.

---

# Business Impact Metrics

## Alerts per Prevented Event

This metric answers: "How many alerts must clinical staff review to prevent one adverse event?"

**Formula**: `Alerts per Prevented Event = 1 / (Precision × Intervention Success Rate)`

**Key assumptions**:
- Not all interventions succeed in preventing the event
- We model this with an `intervention_success_rate` parameter
- If interventions succeed 50% of the time and precision is 0.10, we need 20 alerts per prevented event

This metric helps administrators understand staffing requirements and operational burden.

In [ ]:
intervention_success_rate = 0.5  # Assume 50% of interventions prevent the event

print("Alerts per Prevented Event Analysis:")
print("=" * 70)
print(f"Assumption: {intervention_success_rate:.0%} intervention success rate")
print()
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'Alerts/TP':<15} {'Alerts/Prevented':<15}")
print("-" * 70)

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

for thresh in thresholds:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    prec = precision_score(y_test, y_pred_thresh, zero_division=0)
    rec = recall_score(y_test, y_pred_thresh, zero_division=0)
    
    if prec > 0:
        alerts_per_tp = 1 / prec
        alerts_per_prevented = 1 / (prec * intervention_success_rate)
    else:
        alerts_per_tp = float('inf')
        alerts_per_prevented = float('inf')
    
    print(f"{thresh:<12.1f} {prec:<12.4f} {rec:<12.4f} {alerts_per_tp:<15.1f} {alerts_per_prevented:<15.1f}")

---

## Cost-Benefit Analysis

We can model the costs and benefits of the prediction system to quantify ROI.

**Cost parameters:**

| Outcome | Cost/Benefit | Rationale |
|---------|-------------|----------|
| **True Positive (intervention succeeds)** | -$15,000 (savings) | Prevented hospitalization, emergency care |
| **True Positive (intervention fails)** | $0 | Patient still has event, but we tried |
| **False Positive** | +$50 | Staff time to review alert, contact patient |
| **False Negative** | +$20,000 | Emergency hospitalization, complications |

**Note**: These are illustrative values. Real healthcare organizations should use their own cost data from:
- Claims databases (actual hospitalization costs)
- Time-motion studies (staff review time)
- Literature on intervention effectiveness

In [ ]:
def calculate_cost(y_true, y_pred, costs):
    """
    Calculate total cost of predictions.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        costs: Dict with 'fp', 'fn', 'tp_success', 'intervention_rate'
    
    Returns:
        total_cost: Net cost
        cost_breakdown: Dict with cost components
    """
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Cost calculation
    fp_cost = fp * costs['fp']
    fn_cost = fn * costs['fn']
    tp_savings = tp * costs['intervention_rate'] * costs['tp_success']
    
    total_cost = fp_cost + fn_cost - tp_savings
    
    return total_cost, {
        'fp_cost': fp_cost,
        'fn_cost': fn_cost,
        'tp_savings': tp_savings,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
    }

costs = {
    'fp': 50,           # Cost per false positive (staff review)
    'fn': 20000,        # Cost per missed high-risk patient
    'tp_success': 15000, # Savings per prevented event
    'intervention_rate': 0.5  # Success rate of intervention
}

print("Cost parameters:")
for k, v in costs.items():
    if 'rate' not in k:
        print(f"  {k}: ${v:,.0f}")
    else:
        print(f"  {k}: {v:.0%}")

---

## Cost Analysis by Threshold

We calculate the **net cost** to the healthcare system at each probability threshold.

**Cost components:**
- **FP Cost**: Staff time reviewing false alerts ($50 each)
- **FN Cost**: Missed high-risk patients requiring emergency care ($20,000 each)
- **TP Savings**: Prevented events through early intervention ($15,000 × 50% success rate)

**Net Cost = FP Cost + FN Cost - TP Savings**

The **optimal threshold** minimizes net cost, balancing alert fatigue against missed events.

**Negative net cost** means the model saves money - the value of prevented events exceeds the operational costs.

In [ ]:
print("\nCost Analysis by Threshold:")
print("=" * 80)
print(f"{'Thresh':<8} {'TP':<10} {'FP':<12} {'FN':<10} {'FP Cost':<12} {'FN Cost':<12} {'Savings':<12} {'Net Cost':<12}")
print("-" * 80)

best_threshold = None
best_cost = float('inf')

for thresh in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    total_cost, breakdown = calculate_cost(y_test, y_pred_thresh, costs)
    
    print(f"{thresh:<8.1f} {breakdown['tp']:<10,} {breakdown['fp']:<12,} {breakdown['fn']:<10,} "
          f"${breakdown['fp_cost']:>10,} ${breakdown['fn_cost']:>10,} ${breakdown['tp_savings']:>10,} ${total_cost:>10,.0f}")
    
    if total_cost < best_cost:
        best_cost = total_cost
        best_threshold = thresh

print(f"\nOptimal threshold for cost minimization: {best_threshold}")
print(f"Net cost at optimal threshold: ${best_cost:,.0f}")

---

## Compare to No-Model Baseline

To justify the model's value, we compare to the **no-model baseline**: what if we had no prediction system at all?

**Baseline scenario:**
- No alerts are generated
- All high-risk events are missed (become FN)
- Total cost = number of events × FN cost

**Model savings** = Baseline cost - Model cost at optimal threshold

This ROI calculation helps justify the investment in developing and maintaining the prediction system. It answers the question: "What's the business value of this model compared to doing nothing?"

In [ ]:
n_positives = y_test.sum()
baseline_cost = n_positives * costs['fn']

print(f"\nBaseline (no model): ${baseline_cost:,.0f}")
print(f"Model at optimal threshold: ${best_cost:,.0f}")
print(f"Savings from model: ${baseline_cost - best_cost:,.0f}")

---

## Interpreting Your Results

Based on the metrics above, here's what we can conclude about the model:

**Model Performance:**
- The model's ability to rank patients (ROC-AUC, PR-AUC) tells us if it learned useful patterns
- A ROC-AUC near 0.5 would indicate random guessing; values above 0.8 indicate good discrimination
- PR-AUC should be compared to the baseline (positive class rate)

**Clinical Utility:**
- Precision @ Recall tells us the alert workload for a given sensitivity
- Cost analysis helps justify the model to administrators
- The optimal threshold balances alert fatigue vs missed cases

**Clinical and Business Insights:**
- **Alert workload**: At 90% recall, precision is ~0.04, so staff review ~22 alerts per true positive
- **Cost justification**: Negative net cost means the model provides positive ROI
- **Threshold selection**: Lower thresholds catch more events but increase false alarms

**Next Steps:**
- If performance is poor, revisit feature engineering (you'll add 16 more features in homework)
- Try more sophisticated models (Random Forest, XGBoost)
- Consider additional data sources (clinical notes, more granular lab values)

---

## Stakeholders and Their Priorities

When designing and evaluating a clinical risk prediction system, different stakeholders have different priorities. Understanding these helps us select the right metrics and thresholds.

| Stakeholder | Primary Concerns | Success Metrics |
|-------------|-----------------|-----------------|
| **Chief Medical Officer** | Patient safety, clinical accuracy, interpretability | Adverse event prediction rate, false positive rate |
| **Chief Information Officer** | System reliability, interoperability | Uptime, formatting compliance rate |
| **Chief Financial Officer** | Cost efficiency, ROI of prediction system | Cost per alert, net savings, alerts per prevented event |
| **Compliance Officer** | Regulatory adherence, audit readiness | Zero violations, documentation completeness |
| **Emergency Department Director** | Early warning accuracy, actionable alerts | Alert recall and precision |
| **Nursing Leadership** | Usability, alert fatigue | Interpretability of alerts, false positive alert rate |
| **Patients** | Care quality, accuracy | Outcome improvements |

**Regulatory context**: Under the EU AI Act, clinical risk prediction systems are classified as high-risk AI. This requires documented performance metrics, bias testing, and transparency reporting -- all of which feed into the multi-stakeholder evaluation framework above.

**Key takeaway**: No single metric satisfies all stakeholders. The CMO cares about recall (catching events), nursing leadership cares about precision (alert fatigue), and the compliance officer cares about documentation. A comprehensive evaluation framework must address all perspectives.

---

# Metrics Summary Dashboard

This dashboard consolidates all the metrics we've calculated into a single view for stakeholders.

**Why multiple metrics?**
Different stakeholders care about different things:

| Stakeholder | Key Metrics |
|-------------|-------------|
| **Clinicians** | Recall (sensitivity), false alarm rate, precision |
| **Administrators** | Cost savings, alerts per staff member, ROI |
| **Regulators** | Sensitivity, specificity, performance documentation |
| **Data Scientists** | ROC-AUC, PR-AUC, feature importance |

A comprehensive evaluation reports all relevant metrics so decision-makers can choose the right operating point.

In [ ]:
print("\n" + "=" * 70)
print("EVALUATION METRICS SUMMARY")
print("=" * 70)

print("\n[CLASSIFICATION METRICS]")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"  PR-AUC:  {pr_auc:.4f} (vs {baseline_pr:.4f} random baseline)")

print("\n[PRECISION AT FIXED RECALL]")
for target in [0.70, 0.80, 0.90]:
    prec, _ = precision_at_recall(y_test, y_prob, target)
    alerts = 1 / prec if prec > 0 else float('inf')
    print(f"  Recall={target:.0%}: Precision={prec:.4f} ({alerts:.0f} alerts/TP)")

print("\n[BUSINESS METRICS]")
print(f"  Optimal threshold: {best_threshold}")
print(f"  Net cost at optimal: ${best_cost:,.0f}")
print(f"  Savings vs no model: ${baseline_cost - best_cost:,.0f}")

---

# Summary

## What You Accomplished

| Category | Metrics Implemented |
|----------|---------------------|
| **Classification** | ROC-AUC, PR-AUC, Precision-Recall curve |
| **Threshold Selection** | Precision at Fixed Recall for clinical operating points |
| **Business Impact** | Alerts per prevented event, Cost-benefit analysis |

## Key Insights

1. **Class Imbalance**: With ~3% positive rate, standard accuracy is meaningless
2. **PR-AUC is the key metric**: Shows improvement over random baseline for imbalanced data
3. **Cost Analysis**: Quantifies ROI of the prediction system for administrators
4. **Threshold Selection**: Different thresholds optimize for different objectives (cost vs. recall)

## Model Performance Summary

Based on the 6 Session 1 features:
- **ROC-AUC**: Model's ability to rank positive instances above negative instances
- **PR-AUC significantly above baseline**: Features are predictive of high-risk events
- **Cost analysis**: Model provides positive ROI compared to no-model baseline

In **homework**, you'll implement 8 additional features and use 8 provided features, bringing the total to 22 for potentially better performance.

---

## Next Steps

In **Homework**, you'll:
- Implement 8 additional features using TDD (8 more are provided)
- Re-run baseline comparison with all 22 features
- Evaluate whether additional features improve PR-AUC and cost metrics
- Document performance gains to justify feature engineering effort

---

**Professional Tip**: Always compare PR-AUC to the baseline (positive class rate). A PR-AUC of 0.20 might seem low, but if the baseline is 0.03, that's a ~7x improvement over random guessing.